# Trellis & Trellis 2 — Connection Test

This notebook verifies that:
1. The SSH tunnel to the Lightning AI server is active.
2. The FastAPI inference server is running and healthy.
3. We can successfully invoke inference on both **Trellis** and **Trellis 2** models.

> **Pre-requisite:** Run `local\ssh_connect.bat` in a separate terminal to establish the SSH tunnel before executing these cells.

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys, os, json, base64, requests

# Add the project root so we can import local.remote_infer
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

API_BASE = "http://127.0.0.1:8000"
print(f"Project root : {PROJECT_ROOT}")
print(f"API base URL : {API_BASE}")

Project root : p:\git\setup-gpu-server
API base URL : http://127.0.0.1:8000


In [2]:
# ── Helper: encode a local image to base64 ─────────────────────────────────────
def encode_image(path: str) -> str:
    """Read a local image file and return its base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# ── Set test image path ─────────────────────────────────────────────────────────
# Change this to any local image you want to test with
TEST_IMAGE_PATH = os.path.join(PROJECT_ROOT, "notebooks", "image.png")

if not os.path.exists(TEST_IMAGE_PATH):
    # Create a simple 256x256 solid-color test image if none exists
    try:
        from PIL import Image
        img = Image.new("RGB", (256, 256), color=(100, 149, 237))  # cornflower blue
        img.save(TEST_IMAGE_PATH)
        print(f"✅ Created test image: {TEST_IMAGE_PATH}")
    except ImportError:
        print("❌ Pillow not installed. Please place a test image at:")
        print(f"   {TEST_IMAGE_PATH}")
else:
    print(f"✅ Using existing test image: {TEST_IMAGE_PATH}")

✅ Using existing test image: p:\git\setup-gpu-server\notebooks\image.png


## 1 · Health Check
Ping the `/health` endpoint to confirm the server is reachable through the SSH tunnel.

In [3]:
# ── Health Check ───────────────────────────────────────────────────────────────
try:
    resp = requests.get(f"{API_BASE}/health", timeout=10)
    resp.raise_for_status()
    print("✅ Server is healthy!")
    print(json.dumps(resp.json(), indent=2))
except requests.exceptions.ConnectionError:
    print("❌ Connection failed.")
    print("   → Is the SSH tunnel running?  Run:  local\\ssh_connect.bat <user@host>")
    print("   → Is the server started?      Run:  make start  on the remote machine.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Server is healthy!
{
  "status": "ok",
  "message": "GPU API is healthy"
}


## 2 · Generic Echo Test
Send a simple inference request with a mock model to confirm the full request/response round-trip works.

In [4]:
# ── Generic Echo Test ──────────────────────────────────────────────────────────
payload = {
    "model_id": "echo_test",
    "task_type": "generic",
    "inputs": {"message": "Hello from the notebook!"},
    "parameters": {}
}

try:
    resp = requests.post(f"{API_BASE}/infer", json=payload, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") == "success":
        print("✅ Generic echo test passed!")
        print(json.dumps(data, indent=2))
    else:
        print(f"⚠️  Server returned error: {data.get('error_message')}")
except Exception as e:
    print(f"❌ Echo test failed: {e}")

✅ Generic echo test passed!
{
  "status": "success",
  "result": {
    "status": "success",
    "echo": {
      "message": "Hello from the notebook!"
    },
    "message": "Generic inference executed"
  },
  "error_message": null
}


## 3 · Trellis (v1) — 3D Generation Test
Encode a local image to base64, send it via the `@remote_infer` decorator, and save the returned GLB.

In [5]:
# ── Trellis v1 Test ────────────────────────────────────────────────────────────
from local.remote_infer import remote_infer

@remote_infer(model_id="trellis", task_type="3d_generation")
def trellis_generate_3d(image: str):
    """Placeholder — execution is routed to the remote server."""
    pass

try:
    img_b64 = encode_image(TEST_IMAGE_PATH)
    print(f"Encoded image size: {len(img_b64)} chars")
    result = trellis_generate_3d(image=img_b64)
    print("✅ Trellis v1 inference succeeded!")
    print(f"   Result keys: {list(result.keys())}")

    # Save the GLB file locally
    if "glb_base64" in result:
        out_path = os.path.join(PROJECT_ROOT, "notebooks", "trellis_v1_output.glb")
        with open(out_path, "wb") as f:
            f.write(base64.b64decode(result["glb_base64"]))
        print(f"   Saved GLB to: {out_path}")
except RuntimeError as e:
    print(f"❌ Trellis v1 inference failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

2026-03-03 21:49:13,259 [INFO] Routing call for 'trellis_generate_3d' -> http://127.0.0.1:8000/infer (model: trellis)


Encoded image size: 332692 chars


2026-03-03 21:49:16,231 [ERROR] Server-side error: /io/build/temp.linux-x86_64-cpython-312/spconv/build/core_cc/src/csrc/sparse/all/SpconvOps/SpconvOps_get_indice_pairs.cc(65)
not implemented for CPU ONLY build.


❌ Trellis v1 inference failed: Remote exception: /io/build/temp.linux-x86_64-cpython-312/spconv/build/core_cc/src/csrc/sparse/all/SpconvOps/SpconvOps_get_indice_pairs.cc(65)
not implemented for CPU ONLY build.


## 4 · Trellis 2 — 3D Generation Test
Encode a local image to base64, send it via the `@remote_infer` decorator, and save the returned GLB.

In [9]:
# ── Trellis 2 Test ─────────────────────────────────────────────────────────────
from local.remote_infer import remote_infer


@remote_infer(model_id="trellis2", task_type="3d_generation")
def trellis2_generate_3d(image: str):
    """Placeholder — execution is routed to the remote server."""
    pass

try:
    img_b64 = encode_image(TEST_IMAGE_PATH)
    print(f"Encoded image size: {len(img_b64)} chars")
    result = trellis2_generate_3d(image=img_b64)
    print("✅ Trellis 2 inference succeeded!")
    print(f"   Result keys: {list(result.keys())}")

    # Save the GLB file locally
    if "glb_base64" in result:
        out_path = os.path.join(PROJECT_ROOT, "notebooks", "trellis2_output.glb")
        with open(out_path, "wb") as f:
            f.write(base64.b64decode(result["glb_base64"]))
        print(f"   Saved GLB to: {out_path}")
except RuntimeError as e:
    print(f"❌ Trellis 2 inference failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

2026-03-03 22:38:26,273 [INFO] Routing call for 'trellis2_generate_3d' -> http://127.0.0.1:8000/infer (model: trellis2)


Encoded image size: 332692 chars
✅ Trellis 2 inference succeeded!
   Result keys: ['glb_base64']
   Saved GLB to: p:\git\setup-gpu-server\notebooks\trellis2_output.glb


## 5 · Direct `/infer` POST — Both Models
Bypass the decorator and hit the API directly for a raw round-trip comparison.

In [7]:
# ── Direct POST Test ──────────────────────────────────────────────────────────
img_b64 = encode_image(TEST_IMAGE_PATH)

models_to_test = [
    {"model_id": "trellis",  "label": "Trellis v1"},
    {"model_id": "trellis2", "label": "Trellis 2"},
]

for model in models_to_test:
    payload = {
        "model_id": model["model_id"],
        "task_type": "3d_generation",
        "inputs": {"image": img_b64},
        "parameters": {}
    }
    try:
        resp = requests.post(f"{API_BASE}/infer", json=payload, timeout=600)
        resp.raise_for_status()
        data = resp.json()
        status_icon = "✅" if data.get("status") == "success" else "⚠️"
        print(f"{status_icon} {model['label']}: {data.get('status')}")
        if data.get("status") == "success":
            print(f"   Result keys: {list(data.get('result', {}).keys())}")
        else:
            print(f"   Error: {data.get('error_message')}")
    except Exception as e:
        print(f"❌ {model['label']}: {e}")
    print()

⚠️ Trellis v1: error
   Error: /io/build/temp.linux-x86_64-cpython-312/spconv/build/core_cc/src/csrc/sparse/all/SpconvOps/SpconvOps_get_indice_pairs.cc(65)
not implemented for CPU ONLY build.

⚠️ Trellis 2: error
   Error: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m.
403 Client Error. (Request ID: Root=1-69a79ea1-2e116910620eb1936502d8a5;887ed202-6de6-47c1-8fd2-67b62c5915f4)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m/resolve/main/config.json.
Your request to access model facebook/dinov3-vitl16-pretrain-lvd1689m is awaiting a review from the repo authors.

